# 1. Environment, provenance, and output structure

This cell initializes the computational environment, fixes the random seed, detects whether execution is taking place on a local machine or in Google Colab, creates a run-specific output directory, and records the software/runtime metadata in a machine-readable manifest. This ensures that all downstream results are associated with a single identifiable execution context.

In [1]:
import json
import os
import platform
import random
import sys
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

IN_COLAB = False
try:
    import google.colab  
    from google.colab import drive  
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    DRIVE_ROOT = "/content/drive"
    MOUNT_POINT = f"{DRIVE_ROOT}/MyDrive"
    try:
        if not os.path.exists(MOUNT_POINT):
            drive.mount(DRIVE_ROOT, force_remount=True)
        else:
            drive.mount(DRIVE_ROOT, force_remount=False)
    except Exception as e:
        print(f"[WARN] Drive mount issue: {e}")
    BASE_ROOT = Path(MOUNT_POINT) / "Outputs"
else:
    BASE_ROOT = Path("Outputs")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PROJECT_ROOT = BASE_ROOT / f"bb84_finite_key_phase_study_{RUN_ID}"
FIG_DIR = PROJECT_ROOT / "figures"
TAB_DIR = PROJECT_ROOT / "tables"
OTH_DIR = PROJECT_ROOT / "others"

for directory in (FIG_DIR, TAB_DIR, OTH_DIR):
    directory.mkdir(parents=True, exist_ok=True)

run_manifest = {
    "run_id": RUN_ID,
    "seed": SEED,
    "in_colab": IN_COLAB,
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "matplotlib_version": matplotlib.__version__,
    "project_root": str(PROJECT_ROOT),
    "figure_dir": str(FIG_DIR),
    "table_dir": str(TAB_DIR),
    "other_dir": str(OTH_DIR),
}

manifest_path = OTH_DIR / "run_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, indent=2)

print("[ENV] Colab:", IN_COLAB)
print("[RUN]", RUN_ID)
print("[SEED]", SEED)
print("[ROOT]", PROJECT_ROOT)
print("[MANIFEST]", manifest_path)
print(
    "[VERSIONS]",
    {
        "python": run_manifest["python_version"],
        "numpy": run_manifest["numpy_version"],
        "pandas": run_manifest["pandas_version"],
        "matplotlib": run_manifest["matplotlib_version"],
    },
)

[ENV] Colab: False
[RUN] 20260423_011933
[SEED] 42
[ROOT] Outputs\bb84_finite_key_phase_study_20260423_011933
[MANIFEST] Outputs\bb84_finite_key_phase_study_20260423_011933\others\run_manifest.json
[VERSIONS] {'python': '3.10.20', 'numpy': '2.2.5', 'pandas': '2.3.3', 'matplotlib': '3.10.8'}


# 2. Scope, finite-key setting, and analysis grid

This cell defines the scientific scope of the study and the parameter regime examined throughout the notebook. The analysis is framed as a finite-key BB84-inspired security-margin study under channel disturbance, adversarial interception pressure, and implementation drift. The cell specifies the control grid, the finite sample sizes used in repeated simulations, and the thresholding conventions used for descriptive phase-like summaries. These settings are declared explicitly so that all subsequent estimates can be interpreted against a fixed experimental design.

In [2]:
TIME_HORIZON = 120
T = np.arange(TIME_HORIZON, dtype=float)

EPS = 1e-12

QBER_SECURITY_THRESHOLD = 0.11
PHASE_MARGIN = 0.02

ETA_GRID = np.linspace(0.01, 0.22, 85)
ATTACK_GRID = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40])
DRIFT_GRID = np.array([0.005, 0.010, 0.020, 0.030])

SHOT_COUNTS = np.array([2_000, 5_000, 10_000, 20_000])
DEFAULT_SHOTS = 10_000
N_REPEATS = 250

SCENARIOS = [
    {"name": "low_disturbance", "eta": 0.035, "attack": 0.08, "drift": 0.008},
    {"name": "transition_margin", "eta": 0.075, "attack": 0.16, "drift": 0.015},
    {"name": "elevated_pressure", "eta": 0.105, "attack": 0.24, "drift": 0.020},
    {"name": "high_disturbance", "eta": 0.145, "attack": 0.32, "drift": 0.030},
]

BB84_STAGES = [
    "State preparation",
    "Quantum transmission",
    "Sifting and basis reconciliation",
    "Parameter estimation",
    "Error correction",
    "Privacy amplification",
]

BB84_STAGE_MAP = {
    "State preparation": "Source fidelity and preparation stability.",
    "Quantum transmission": "Channel disturbance and intercept pressure.",
    "Sifting and basis reconciliation": "Observed mismatch after basis filtering.",
    "Parameter estimation": "Finite-key inference for QBER and secrecy margin.",
    "Error correction": "Leakage burden induced by observed disturbance.",
    "Privacy amplification": "Residual extractable secrecy after finite-key penalties.",
}

print("[CONFIG] time horizon:", TIME_HORIZON)
print("[CONFIG] eta range:", (float(ETA_GRID.min()), float(ETA_GRID.max())))
print("[CONFIG] attack levels:", ATTACK_GRID.tolist())
print("[CONFIG] drift levels:", DRIFT_GRID.tolist())
print("[CONFIG] shot counts:", SHOT_COUNTS.tolist())
print("[CONFIG] repeats:", N_REPEATS)
print("[CONFIG] qber threshold:", QBER_SECURITY_THRESHOLD)
print("[BB84] stages:", " | ".join(BB84_STAGES))

[CONFIG] time horizon: 120
[CONFIG] eta range: (0.01, 0.22)
[CONFIG] attack levels: [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]
[CONFIG] drift levels: [0.005, 0.01, 0.02, 0.03]
[CONFIG] shot counts: [2000, 5000, 10000, 20000]
[CONFIG] repeats: 250
[CONFIG] qber threshold: 0.11
[BB84] stages: State preparation | Quantum transmission | Sifting and basis reconciliation | Parameter estimation | Error correction | Privacy amplification


# 3. Finite-key security model and descriptive phase metrics

This cell defines the core functions used throughout the study. The model treats the observed quantum bit error rate as a stochastic finite-sample quantity shaped by channel disturbance, adversarial interception pressure, implementation drift, and weak temporal modulation. From the observed error rate, the cell computes a finite-key-inspired secrecy margin, a leakage burden proxy, and a descriptive order parameter used only for phase-like summaries. The definitions are chosen so that each quantity remains interpretable in cryptographic terms and can be stress-tested under repeated simulation.

In [4]:
def clip01(x):
    return np.clip(x, 0.0, 1.0)


def binary_entropy(p):
    p = np.clip(p, EPS, 1.0 - EPS)
    return -(p * np.log2(p) + (1.0 - p) * np.log2(1.0 - p))


def deterministic_qber_mean(eta, attack, drift, t, time_horizon=TIME_HORIZON):
    seasonal = 0.006 * np.sin(2.0 * np.pi * t / max(time_horizon, 1))
    trend = 0.010 * (t / max(time_horizon - 1, 1))
    qber_mean = eta + 0.055 * attack + 0.90 * drift + seasonal + trend
    return float(clip01(qber_mean))


def effective_detection_count(shots, attack, drift):
    retention = 1.0 - 0.18 * attack - 1.10 * drift
    retention = float(np.clip(retention, 0.35, 1.0))
    return max(50, int(round(shots * retention)))


def sample_observed_qber(qber_mean, shots, attack, drift, rng):
    n_eff = effective_detection_count(shots, attack=attack, drift=drift)
    errors = rng.binomial(n=n_eff, p=float(np.clip(qber_mean, EPS, 1.0 - EPS)))
    return float(errors / n_eff), n_eff


def finite_key_penalty(shots):
    return float(2.6 / np.sqrt(max(shots, 1)))


def secrecy_margin(qber_obs, shots, drift):
    margin = 1.0 - 2.0 * binary_entropy(np.minimum(qber_obs, 0.499999)) - finite_key_penalty(shots) - 1.35 * drift
    return float(margin)


def leakage_burden(qber_obs, attack, drift):
    leakage = 0.65 * attack + 0.75 * qber_obs + 0.90 * drift
    return float(leakage)


def order_parameter_from_margin(margin):
    scale = max(abs(QBER_SECURITY_THRESHOLD), EPS)
    return float(margin / scale)


def classify_phase(psi, margin=PHASE_MARGIN):
    if psi > margin:
        return "secure"
    if psi < -margin:
        return "insecure"
    return "transition"


def susceptibility(values, controls):
    return np.gradient(np.asarray(values, dtype=float), np.asarray(controls, dtype=float))


def first_transition_time(psi_series):
    idx = np.where(np.asarray(psi_series) <= 0.0)[0]
    return int(idx[0]) if len(idx) else -1


rng_smoke = np.random.default_rng(SEED)
qber_mean_smoke = deterministic_qber_mean(eta=0.06, attack=0.10, drift=0.01, t=15)
qber_obs_smoke, n_eff_smoke = sample_observed_qber(
    qber_mean_smoke,
    shots=DEFAULT_SHOTS,
    attack=0.10,
    drift=0.01,
    rng=rng_smoke,
)
margin_smoke = secrecy_margin(qber_obs_smoke, shots=n_eff_smoke, drift=0.01)
psi_smoke = order_parameter_from_margin(margin_smoke)

print("[SMOKE] mean qber =", round(qber_mean_smoke, 6))
print("[SMOKE] observed qber =", round(qber_obs_smoke, 6))
print("[SMOKE] effective detections =", n_eff_smoke)
print("[SMOKE] secrecy margin =", round(margin_smoke, 6))
print("[SMOKE] psi =", round(psi_smoke, 6))
print("[SMOKE] phase =", classify_phase(psi_smoke))

[SMOKE] mean qber = 0.080003
[SMOKE] observed qber = 0.077137
[SMOKE] effective detections = 9710
[SMOKE] secrecy margin = 0.176095
[SMOKE] psi = 1.600862
[SMOKE] phase = secure


# 4. Time-resolved stochastic trajectory simulator

This cell defines the trajectory simulator used for scenario analysis and time-resolved diagnostics. For a fixed disturbance level, attack pressure, implementation drift, and sample size, the simulator generates a stochastic sequence of observed QBER values and derives the corresponding secrecy margin, leakage burden, descriptive phase indicator, and temporal sensitivity summaries. The resulting trajectory is returned in tabular form, allowing both aggregate statistics and figures to be produced from a common representation.

In [5]:
def simulate_trajectory(eta, attack, drift, shots=DEFAULT_SHOTS, time_horizon=TIME_HORIZON, seed=None):
    rng = np.random.default_rng(SEED if seed is None else seed)

    tt = np.arange(time_horizon, dtype=float)
    qber_mean = np.zeros(time_horizon, dtype=float)
    qber_obs = np.zeros(time_horizon, dtype=float)
    n_eff = np.zeros(time_horizon, dtype=int)
    margin = np.zeros(time_horizon, dtype=float)
    leakage = np.zeros(time_horizon, dtype=float)

    for k in range(time_horizon):
        qber_mean[k] = deterministic_qber_mean(
            eta=eta,
            attack=attack,
            drift=drift,
            t=k,
            time_horizon=time_horizon,
        )
        qber_obs[k], n_eff[k] = sample_observed_qber(
            qber_mean=qber_mean[k],
            shots=shots,
            attack=attack,
            drift=drift,
            rng=rng,
        )
        margin[k] = secrecy_margin(qber_obs=qber_obs[k], shots=n_eff[k], drift=drift)
        leakage[k] = leakage_burden(qber_obs=qber_obs[k], attack=attack, drift=drift)

    psi = np.array([order_parameter_from_margin(x) for x in margin], dtype=float)
    dpsi_dt = np.gradient(psi, tt)
    dqber_dt = np.gradient(qber_obs, tt)

    return pd.DataFrame(
        {
            "t": tt,
            "eta": float(eta),
            "attack": float(attack),
            "drift": float(drift),
            "shots": int(shots),
            "n_eff": n_eff,
            "qber_mean": qber_mean,
            "qber_obs": qber_obs,
            "secrecy_margin": margin,
            "leakage_burden": leakage,
            "psi": psi,
            "dpsi_dt": dpsi_dt,
            "dqber_dt": dqber_dt,
            "phase": [classify_phase(x) for x in psi],
        }
    )


smoke_df = simulate_trajectory(
    eta=0.07,
    attack=0.14,
    drift=0.012,
    shots=DEFAULT_SHOTS,
    seed=SEED,
)

print(smoke_df.head(3).to_string(index=False))
print("[SMOKE] first transition time =", first_transition_time(smoke_df["psi"].values))
print("[SMOKE] final phase =", smoke_df["phase"].iloc[-1])

  t  eta  attack  drift  shots  n_eff  qber_mean  qber_obs  secrecy_margin  leakage_burden      psi   dpsi_dt  dqber_dt  phase
0.0 0.07    0.14  0.012  10000   9616   0.088500  0.085483        0.114869        0.165912 1.044262 -0.287319  0.004680 secure
1.0 0.07    0.14  0.012  10000   9616   0.088898  0.090162        0.083264        0.169422 0.756943  0.042178 -0.000676 secure
2.0 0.07    0.14  0.012  10000   9616   0.089295  0.084131        0.124148        0.164898 1.128618  0.066595 -0.001092 secure
[SMOKE] first transition time = -1
[SMOKE] final phase = secure


# 5. Repeated endpoint sweep over disturbance, attack pressure, and drift

This cell evaluates the terminal security state over the full control grid using repeated finite-sample simulations. For each combination of disturbance level, attack pressure, and implementation drift, the cell generates repeated endpoint realizations, computes the observed QBER, secrecy margin, leakage burden, and descriptive phase label, and then aggregates these outcomes into uncertainty-aware summaries. The resulting tables provide the basis for phase-boundary estimation, confidence intervals, and robustness analysis in later cells.

In [6]:
def empirical_interval(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    lo, med, hi = np.quantile(values, [alpha / 2.0, 0.5, 1.0 - alpha / 2.0])
    return float(lo), float(med), float(hi)


def simulate_endpoint_replicates(
    eta,
    attack,
    drift,
    shots=DEFAULT_SHOTS,
    n_repeats=N_REPEATS,
    t_eval=TIME_HORIZON - 1,
    rng=None,
):
    rng = np.random.default_rng(SEED) if rng is None else rng

    qber_mean = deterministic_qber_mean(
        eta=eta,
        attack=attack,
        drift=drift,
        t=t_eval,
        time_horizon=TIME_HORIZON,
    )
    n_eff = effective_detection_count(shots, attack=attack, drift=drift)
    errors = rng.binomial(
        n=n_eff,
        p=float(np.clip(qber_mean, EPS, 1.0 - EPS)),
        size=n_repeats,
    )
    qber_obs = errors / n_eff
    margins = np.array([secrecy_margin(q, shots=n_eff, drift=drift) for q in qber_obs], dtype=float)
    leakage = np.array([leakage_burden(q, attack=attack, drift=drift) for q in qber_obs], dtype=float)
    psi = np.array([order_parameter_from_margin(m) for m in margins], dtype=float)
    phase = np.array([classify_phase(x) for x in psi], dtype=object)

    return pd.DataFrame(
        {
            "eta": float(eta),
            "attack": float(attack),
            "drift": float(drift),
            "shots": int(shots),
            "t_eval": int(t_eval),
            "repeat": np.arange(n_repeats, dtype=int),
            "n_eff": int(n_eff),
            "qber_mean": float(qber_mean),
            "qber_obs": qber_obs,
            "secrecy_margin": margins,
            "leakage_burden": leakage,
            "psi": psi,
            "phase": phase,
        }
    )


rng_panel = np.random.default_rng(SEED)
endpoint_frames = []
summary_rows = []

for drift in DRIFT_GRID:
    for attack in ATTACK_GRID:
        for eta in ETA_GRID:
            df = simulate_endpoint_replicates(
                eta=float(eta),
                attack=float(attack),
                drift=float(drift),
                shots=DEFAULT_SHOTS,
                n_repeats=N_REPEATS,
                rng=rng_panel,
            )
            endpoint_frames.append(df)

            qber_lo, qber_med, qber_hi = empirical_interval(df["qber_obs"].values)
            margin_lo, margin_med, margin_hi = empirical_interval(df["secrecy_margin"].values)
            psi_lo, psi_med, psi_hi = empirical_interval(df["psi"].values)

            phase_counts = df["phase"].value_counts()
            p_secure = float((df["phase"] == "secure").mean())
            p_transition = float((df["phase"] == "transition").mean())
            p_insecure = float((df["phase"] == "insecure").mean())
            dominant_phase = phase_counts.idxmax()

            summary_rows.append(
                {
                    "eta": float(eta),
                    "attack": float(attack),
                    "drift": float(drift),
                    "shots": int(DEFAULT_SHOTS),
                    "n_repeats": int(N_REPEATS),
                    "n_eff": int(df["n_eff"].iloc[0]),
                    "qber_mean_det": float(df["qber_mean"].iloc[0]),
                    "qber_obs_mean": float(df["qber_obs"].mean()),
                    "qber_obs_sd": float(df["qber_obs"].std(ddof=1)),
                    "qber_obs_ci_low": qber_lo,
                    "qber_obs_ci_med": qber_med,
                    "qber_obs_ci_high": qber_hi,
                    "margin_mean": float(df["secrecy_margin"].mean()),
                    "margin_sd": float(df["secrecy_margin"].std(ddof=1)),
                    "margin_ci_low": margin_lo,
                    "margin_ci_med": margin_med,
                    "margin_ci_high": margin_hi,
                    "psi_mean": float(df["psi"].mean()),
                    "psi_sd": float(df["psi"].std(ddof=1)),
                    "psi_ci_low": psi_lo,
                    "psi_ci_med": psi_med,
                    "psi_ci_high": psi_hi,
                    "mean_leakage": float(df["leakage_burden"].mean()),
                    "p_secure": p_secure,
                    "p_transition": p_transition,
                    "p_insecure": p_insecure,
                    "dominant_phase": dominant_phase,
                }
            )

endpoint_df = pd.concat(endpoint_frames, ignore_index=True)
endpoint_summary_df = pd.DataFrame(summary_rows).sort_values(["drift", "attack", "eta"]).reset_index(drop=True)

endpoint_csv = TAB_DIR / "endpoint_replicates.csv"
endpoint_summary_csv = TAB_DIR / "endpoint_summary.csv"

endpoint_df.to_csv(endpoint_csv, index=False)
endpoint_summary_df.to_csv(endpoint_summary_csv, index=False)

print("[ENDPOINT] replicate rows =", len(endpoint_df))
print("[ENDPOINT] summary rows =", len(endpoint_summary_df))
print("[TABLE] saved:", endpoint_csv)
print("[TABLE] saved:", endpoint_summary_csv)
print(endpoint_summary_df.head(8).to_string(index=False))

[ENDPOINT] replicate rows = 680000
[ENDPOINT] summary rows = 2720
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\endpoint_replicates.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\endpoint_summary.csv
   eta  attack  drift  shots  n_repeats  n_eff  qber_mean_det  qber_obs_mean  qber_obs_sd  qber_obs_ci_low  qber_obs_ci_med  qber_obs_ci_high  margin_mean  margin_sd  margin_ci_low  margin_ci_med  margin_ci_high  psi_mean   psi_sd  psi_ci_low  psi_ci_med  psi_ci_high  mean_leakage  p_secure  p_transition  p_insecure dominant_phase
0.0100    0.05  0.005  10000        250   9855       0.026936       0.026748     0.001646         0.023384         0.026738          0.029934     0.611588   0.017093       0.578937       0.611544        0.646978  5.559894 0.155391    5.263068    5.559488     5.881615      0.057061       1.0           0.0         0.0         secure
0.0125    0.05  0.005  10000        250   9855       0.029436       0.02944

# 6. Operational phase-boundary estimation with uncertainty

This cell estimates the disturbance boundary at which the terminal security state changes from predominantly secure to predominantly insecure. For each combination of attack pressure and implementation drift, the boundary is defined in two complementary ways: the zero crossing of the mean secrecy margin and the disturbance level at which the empirical secure-state probability falls to one half. The latter is accompanied by a bootstrap interval obtained from the endpoint replicates, yielding an uncertainty-aware phase-boundary estimate suitable for figures and summary tables.

In [7]:
def interpolate_crossing(x, y, target):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float) - float(target)

    exact = np.where(np.isclose(y, 0.0, atol=1e-12))[0]
    if len(exact):
        return float(x[int(exact[0])])

    crossing_idx = np.where(y[:-1] * y[1:] < 0.0)[0]
    if len(crossing_idx) == 0:
        return np.nan

    i = int(crossing_idx[0])
    x0, x1 = x[i], x[i + 1]
    y0, y1 = y[i], y[i + 1]
    return float(x0 - y0 * (x1 - x0) / (y1 - y0))


def bootstrap_eta_c(secure_matrix, eta_values, n_boot=200, seed=SEED):
    secure_matrix = np.asarray(secure_matrix, dtype=float)
    eta_values = np.asarray(eta_values, dtype=float)

    n_eta, n_rep = secure_matrix.shape
    rng = np.random.default_rng(seed)
    eta_boot = []

    row_index = np.arange(n_eta)[:, None]

    for _ in range(n_boot):
        draw_index = rng.integers(0, n_rep, size=(n_eta, n_rep))
        p_secure_boot = secure_matrix[row_index, draw_index].mean(axis=1)
        eta_c_boot = interpolate_crossing(eta_values, p_secure_boot, target=0.5)
        if np.isfinite(eta_c_boot):
            eta_boot.append(float(eta_c_boot))

    if len(eta_boot) == 0:
        return np.nan, np.nan, np.nan, 0

    eta_boot = np.asarray(eta_boot, dtype=float)
    lo, med, hi = np.quantile(eta_boot, [0.025, 0.5, 0.975])
    return float(lo), float(med), float(hi), int(len(eta_boot))


boundary_rows = []
group_seed_rng = np.random.default_rng(SEED)

for drift in DRIFT_GRID:
    for attack in ATTACK_GRID:
        sub = endpoint_summary_df[
            (endpoint_summary_df["drift"] == float(drift)) &
            (endpoint_summary_df["attack"] == float(attack))
        ].sort_values("eta")

        eta_values = sub["eta"].to_numpy(dtype=float)
        p_secure_values = sub["p_secure"].to_numpy(dtype=float)
        margin_values = sub["margin_mean"].to_numpy(dtype=float)

        eta_c_margin = interpolate_crossing(eta_values, margin_values, target=0.0)
        eta_c_psecure50 = interpolate_crossing(eta_values, p_secure_values, target=0.5)

        secure_rows = []
        for eta in eta_values:
            reps = endpoint_df[
                (endpoint_df["drift"] == float(drift)) &
                (endpoint_df["attack"] == float(attack)) &
                (endpoint_df["eta"] == float(eta))
            ].sort_values("repeat")
            secure_rows.append((reps["phase"] == "secure").to_numpy(dtype=float))

        secure_matrix = np.vstack(secure_rows)
        boot_seed = int(group_seed_rng.integers(0, 2**32 - 1))
        eta_c_low, eta_c_med, eta_c_high, n_boot_valid = bootstrap_eta_c(
            secure_matrix=secure_matrix,
            eta_values=eta_values,
            n_boot=200,
            seed=boot_seed,
        )

        dp_deta = susceptibility(p_secure_values, eta_values)
        eta_at_max_slope = float(eta_values[int(np.argmax(np.abs(dp_deta)))])

        boundary_rows.append(
            {
                "attack": float(attack),
                "drift": float(drift),
                "eta_c_margin": eta_c_margin,
                "eta_c_psecure50": eta_c_psecure50,
                "eta_c_boot_low": eta_c_low,
                "eta_c_boot_med": eta_c_med,
                "eta_c_boot_high": eta_c_high,
                "n_boot_valid": int(n_boot_valid),
                "p_secure_min_eta": float(p_secure_values[0]),
                "p_secure_max_eta": float(p_secure_values[-1]),
                "max_abs_dp_deta": float(np.max(np.abs(dp_deta))),
                "eta_at_max_abs_dp_deta": eta_at_max_slope,
            }
        )

boundary_df = pd.DataFrame(boundary_rows).sort_values(["drift", "attack"]).reset_index(drop=True)

boundary_csv = TAB_DIR / "phase_boundary_summary.csv"
boundary_df.to_csv(boundary_csv, index=False)

plt.figure(figsize=(8, 5.2))
for drift in DRIFT_GRID:
    sub = boundary_df[boundary_df["drift"] == float(drift)].sort_values("attack")
    x = sub["attack"].to_numpy(dtype=float)
    y = sub["eta_c_boot_med"].to_numpy(dtype=float)
    ylo = sub["eta_c_boot_low"].to_numpy(dtype=float)
    yhi = sub["eta_c_boot_high"].to_numpy(dtype=float)

    plt.plot(x, y, marker="o", label=f"drift={drift:.3f}")
    plt.fill_between(x, ylo, yhi, alpha=0.18)

plt.xlabel("Attack pressure")
plt.ylabel(r"Estimated boundary $\eta_c$")
plt.title("Operational phase boundary with bootstrap uncertainty")
plt.legend()
plt.tight_layout()

boundary_fig = FIG_DIR / "phase_boundary_with_uncertainty.png"
plt.savefig(boundary_fig, dpi=240)
plt.close()

print("[BOUNDARY] rows =", len(boundary_df))
print("[TABLE] saved:", boundary_csv)
print("[FIGURE] saved:", boundary_fig)
print(boundary_df.head(10).to_string(index=False))

[BOUNDARY] rows = 32
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\phase_boundary_summary.csv
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\figures\phase_boundary_with_uncertainty.png
 attack  drift  eta_c_margin  eta_c_psecure50  eta_c_boot_low  eta_c_boot_med  eta_c_boot_high  n_boot_valid  p_secure_min_eta  p_secure_max_eta  max_abs_dp_deta  eta_at_max_abs_dp_deta
   0.05  0.005      0.087717         0.087237        0.086738        0.087252         0.087781           200               1.0               0.0            104.0                  0.0875
   0.10  0.005      0.084732         0.084412        0.084133        0.084427         0.084922           200               1.0               0.0            113.6                  0.0850
   0.15  0.005      0.082298         0.082045        0.081699        0.082055         0.082466           200               1.0               0.0            126.4                  0.0825
   0.20  0.005      0.

# 7. Disturbance-response profiles and uncertainty-aware susceptibility summaries

This cell characterizes how the terminal security state changes across the disturbance sweep for representative attack-pressure and drift settings. For each fixed drift level, the cell visualizes the secure-state probability as a function of disturbance and overlays the operational boundary estimate. It also summarizes the slope of the secure-probability curve with respect to disturbance, yielding a descriptive susceptibility profile that identifies where the transition is most abrupt. These figures are intended to complement the boundary table by showing both the location and the sharpness of the transition.

In [8]:
profile_specs = [
    {"attack": 0.10, "drift": 0.005},
    {"attack": 0.20, "drift": 0.010},
    {"attack": 0.30, "drift": 0.020},
    {"attack": 0.35, "drift": 0.030},
]

profile_rows = []

for spec in profile_specs:
    attack = float(spec["attack"])
    drift = float(spec["drift"])

    sub = endpoint_summary_df[
        (endpoint_summary_df["attack"] == attack) &
        (endpoint_summary_df["drift"] == drift)
    ].sort_values("eta").copy()

    sub["dp_secure_deta"] = susceptibility(sub["p_secure"].values, sub["eta"].values)
    sub["profile_attack"] = attack
    sub["profile_drift"] = drift
    profile_rows.append(sub)

profiles_df = pd.concat(profile_rows, ignore_index=True)
profiles_csv = TAB_DIR / "transition_profiles.csv"
profiles_df.to_csv(profiles_csv, index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for drift in DRIFT_GRID:
    attack = 0.20
    sub = endpoint_summary_df[
        (endpoint_summary_df["attack"] == attack) &
        (endpoint_summary_df["drift"] == float(drift))
    ].sort_values("eta")

    boundary_row = boundary_df[
        (boundary_df["attack"] == attack) &
        (boundary_df["drift"] == float(drift))
    ].iloc[0]

    axes[0].plot(
        sub["eta"],
        sub["p_secure"],
        marker="o",
        ms=3.5,
        lw=1.8,
        label=f"drift={drift:.3f}",
    )
    axes[0].axvline(
        boundary_row["eta_c_boot_med"],
        linestyle="--",
        linewidth=1.2,
    )

axes[0].set_xlabel(r"Disturbance $\eta$")
axes[0].set_ylabel("Secure-state probability")
axes[0].set_title(r"Secure-state probability profiles at fixed attack ($a=0.20$)")
axes[0].legend()

for spec in profile_specs:
    attack = float(spec["attack"])
    drift = float(spec["drift"])

    sub = profiles_df[
        (profiles_df["profile_attack"] == attack) &
        (profiles_df["profile_drift"] == drift)
    ].sort_values("eta")

    axes[1].plot(
        sub["eta"],
        np.abs(sub["dp_secure_deta"]),
        lw=2.0,
        label=f"a={attack:.2f}, d={drift:.3f}",
    )

axes[1].set_xlabel(r"Disturbance $\eta$")
axes[1].set_ylabel(r"$\left|\partial p_{\mathrm{secure}} / \partial \eta\right|$")
axes[1].set_title("Transition sharpness across representative settings")
axes[1].legend()

plt.tight_layout()

profiles_fig = FIG_DIR / "transition_profiles_and_susceptibility.png"
plt.savefig(profiles_fig, dpi=240)
plt.close()

print("[PROFILE] rows =", len(profiles_df))
print("[TABLE] saved:", profiles_csv)
print("[FIGURE] saved:", profiles_fig)
print(
    profiles_df[
        [
            "profile_attack",
            "profile_drift",
            "eta",
            "p_secure",
            "dp_secure_deta",
            "dominant_phase",
        ]
    ].head(12).to_string(index=False)
)

[PROFILE] rows = 340
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\transition_profiles.csv
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\figures\transition_profiles_and_susceptibility.png
 profile_attack  profile_drift    eta  p_secure  dp_secure_deta dominant_phase
            0.1          0.005 0.0100       1.0             0.0         secure
            0.1          0.005 0.0125       1.0             0.0         secure
            0.1          0.005 0.0150       1.0             0.0         secure
            0.1          0.005 0.0175       1.0             0.0         secure
            0.1          0.005 0.0200       1.0             0.0         secure
            0.1          0.005 0.0225       1.0             0.0         secure
            0.1          0.005 0.0250       1.0             0.0         secure
            0.1          0.005 0.0275       1.0             0.0         secure
            0.1          0.005 0.0300       1.0    

# 8. Shot-count dependence of the operational boundary

This cell evaluates how the estimated disturbance boundary changes with finite sample size. For each shot-count setting, the analysis repeats the endpoint sweep, estimates the secure-state probability boundary, and compares the resulting boundary location across attack-pressure and drift conditions. This isolates the effect of finite-key sample size on transition location and provides a direct robustness check against one of the most important design parameters in the study.

In [10]:
shot_boundary_rows = []

for shots in SHOT_COUNTS:
    shot_seed_rng = np.random.default_rng(SEED + int(shots))
    shot_summary_rows = []

    for drift in DRIFT_GRID:
        for attack in ATTACK_GRID:
            for eta in ETA_GRID:
                df = simulate_endpoint_replicates(
                    eta=float(eta),
                    attack=float(attack),
                    drift=float(drift),
                    shots=int(shots),
                    n_repeats=N_REPEATS,
                    rng=shot_seed_rng,
                )

                shot_summary_rows.append(
                    {
                        "eta": float(eta),
                        "attack": float(attack),
                        "drift": float(drift),
                        "shots": int(shots),
                        "p_secure": float((df["phase"] == "secure").mean()),
                        "margin_mean": float(df["secrecy_margin"].mean()),
                    }
                )

    shot_summary_df = (
        pd.DataFrame(shot_summary_rows)
        .sort_values(["drift", "attack", "eta"])
        .reset_index(drop=True)
    )

    for drift in DRIFT_GRID:
        for attack in ATTACK_GRID:
            sub = shot_summary_df[
                (shot_summary_df["drift"] == float(drift)) &
                (shot_summary_df["attack"] == float(attack))
            ].sort_values("eta")

            eta_values = sub["eta"].to_numpy(dtype=float)
            p_secure_values = sub["p_secure"].to_numpy(dtype=float)
            margin_values = sub["margin_mean"].to_numpy(dtype=float)

            eta_c_margin = interpolate_crossing(eta_values, margin_values, target=0.0)
            eta_c_psecure50 = interpolate_crossing(eta_values, p_secure_values, target=0.5)

            shot_boundary_rows.append(
                {
                    "shots": int(shots),
                    "attack": float(attack),
                    "drift": float(drift),
                    "eta_c_margin": eta_c_margin,
                    "eta_c_psecure50": eta_c_psecure50,
                }
            )

shot_boundary_df = (
    pd.DataFrame(shot_boundary_rows)
    .sort_values(["shots", "drift", "attack"])
    .reset_index(drop=True)
)

shot_boundary_csv = TAB_DIR / "shot_count_boundary_summary.csv"
shot_boundary_df.to_csv(shot_boundary_csv, index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for drift in [0.005, 0.020]:
    sub = shot_boundary_df[shot_boundary_df["drift"] == drift]
    for shots in SHOT_COUNTS:
        ssub = sub[sub["shots"] == int(shots)].sort_values("attack")
        axes[0].plot(
            ssub["attack"],
            ssub["eta_c_psecure50"],
            marker="o",
            lw=1.8,
            label=f"{shots} shots, drift={drift:.3f}",
        )

axes[0].set_xlabel("Attack pressure")
axes[0].set_ylabel(r"Estimated boundary $\eta_c$")
axes[0].set_title("Boundary location versus shot count")
axes[0].legend(fontsize=9)

agg = (
    shot_boundary_df
    .groupby("shots", as_index=False)
    .agg(
        eta_c_mean=("eta_c_psecure50", "mean"),
        eta_c_sd=("eta_c_psecure50", "std"),
    )
)

axes[1].errorbar(
    agg["shots"],
    agg["eta_c_mean"],
    yerr=agg["eta_c_sd"],
    marker="o",
    lw=1.8,
    capsize=4,
)
axes[1].set_xlabel("Shot count")
axes[1].set_ylabel(r"Mean estimated boundary $\eta_c$")
axes[1].set_title("Global boundary trend across finite sample sizes")

plt.tight_layout()

shot_boundary_fig = FIG_DIR / "shot_count_boundary_dependence.png"
plt.savefig(shot_boundary_fig, dpi=240)
plt.close()

print("[SHOT] rows =", len(shot_boundary_df))
print("[TABLE] saved:", shot_boundary_csv)
print("[FIGURE] saved:", shot_boundary_fig)
print(shot_boundary_df.head(12).to_string(index=False))
print(agg.to_string(index=False))

[SHOT] rows = 128
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\shot_count_boundary_summary.csv
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\figures\shot_count_boundary_dependence.png
 shots  attack  drift  eta_c_margin  eta_c_psecure50
  2000    0.05  0.005      0.083020         0.082727
  2000    0.10  0.005      0.080332         0.079342
  2000    0.15  0.005      0.076978         0.076694
  2000    0.20  0.005      0.074493         0.073906
  2000    0.25  0.005      0.071132         0.070278
  2000    0.30  0.005      0.068506         0.068590
  2000    0.35  0.005      0.065571         0.065278
  2000    0.40  0.005      0.062828         0.062586
  2000    0.05  0.010      0.077034         0.077105
  2000    0.10  0.010      0.074572         0.073833
  2000    0.15  0.010      0.071798         0.071324
  2000    0.20  0.010      0.068470         0.068125
 shots  eta_c_mean  eta_c_sd
  2000    0.059951  0.012655
  5000    0.063443

# 9. Sensitivity of the operational boundary to the descriptive phase margin

This cell evaluates how strongly the boundary estimates depend on the descriptive phase-margin convention used to classify terminal states as secure, transition, or insecure. Because the endpoint replicates already store the normalized security indicator, the classification can be recomputed without rerunning the stochastic simulation. The resulting analysis quantifies whether the operational boundary is robust to moderate changes in the descriptive transition band and separates substantive boundary behavior from threshold-selection artifacts.

In [11]:
PHASE_MARGIN_GRID = np.array([0.01, 0.02, 0.03, 0.05], dtype=float)

phase_margin_profile_frames = []
phase_margin_boundary_rows = []

for phase_margin in PHASE_MARGIN_GRID:
    temp = endpoint_df.loc[:, ["eta", "attack", "drift"]].copy()
    psi_values = endpoint_df["psi"].to_numpy(dtype=float)

    temp["p_secure"] = (psi_values > phase_margin).astype(float)
    temp["p_transition"] = (np.abs(psi_values) <= phase_margin).astype(float)
    temp["p_insecure"] = (psi_values < -phase_margin).astype(float)

    grouped = (
        temp.groupby(["drift", "attack", "eta"], as_index=False)[["p_secure", "p_transition", "p_insecure"]]
        .mean()
        .sort_values(["drift", "attack", "eta"])
        .reset_index(drop=True)
    )
    grouped["phase_margin"] = float(phase_margin)
    phase_margin_profile_frames.append(grouped)

    for drift in DRIFT_GRID:
        for attack in ATTACK_GRID:
            sub = grouped[
                (grouped["drift"] == float(drift)) &
                (grouped["attack"] == float(attack))
            ].sort_values("eta")

            eta_values = sub["eta"].to_numpy(dtype=float)
            p_secure_values = sub["p_secure"].to_numpy(dtype=float)

            eta_c = interpolate_crossing(eta_values, p_secure_values, target=0.5)
            dp_deta = susceptibility(p_secure_values, eta_values)

            phase_margin_boundary_rows.append(
                {
                    "phase_margin": float(phase_margin),
                    "attack": float(attack),
                    "drift": float(drift),
                    "eta_c_psecure50": eta_c,
                    "max_abs_dp_deta": float(np.max(np.abs(dp_deta))),
                    "eta_at_max_abs_dp_deta": float(eta_values[int(np.argmax(np.abs(dp_deta)))]),
                }
            )

phase_margin_profiles_df = pd.concat(phase_margin_profile_frames, ignore_index=True)
phase_margin_boundary_df = (
    pd.DataFrame(phase_margin_boundary_rows)
    .sort_values(["phase_margin", "drift", "attack"])
    .reset_index(drop=True)
)

phase_margin_profiles_csv = TAB_DIR / "phase_margin_profiles.csv"
phase_margin_boundary_csv = TAB_DIR / "phase_margin_boundary_summary.csv"

phase_margin_profiles_df.to_csv(phase_margin_profiles_csv, index=False)
phase_margin_boundary_df.to_csv(phase_margin_boundary_csv, index=False)

baseline_margin = 0.02
baseline = (
    phase_margin_boundary_df[phase_margin_boundary_df["phase_margin"] == baseline_margin]
    .loc[:, ["attack", "drift", "eta_c_psecure50"]]
    .rename(columns={"eta_c_psecure50": "eta_c_baseline"})
)

delta_df = phase_margin_boundary_df.merge(baseline, on=["attack", "drift"], how="left")
delta_df["eta_c_delta"] = delta_df["eta_c_psecure50"] - delta_df["eta_c_baseline"]

delta_summary = (
    delta_df.groupby("phase_margin", as_index=False)
    .agg(
        mean_eta_c=("eta_c_psecure50", "mean"),
        mean_delta=("eta_c_delta", "mean"),
        sd_delta=("eta_c_delta", "std"),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

selected_drift = 0.010
for phase_margin in PHASE_MARGIN_GRID:
    sub = phase_margin_boundary_df[
        (phase_margin_boundary_df["phase_margin"] == float(phase_margin)) &
        (phase_margin_boundary_df["drift"] == selected_drift)
    ].sort_values("attack")

    axes[0].plot(
        sub["attack"],
        sub["eta_c_psecure50"],
        marker="o",
        lw=1.8,
        label=f"margin={phase_margin:.2f}",
    )

axes[0].set_xlabel("Attack pressure")
axes[0].set_ylabel(r"Estimated boundary $\eta_c$")
axes[0].set_title(r"Boundary sensitivity at fixed drift ($d=0.010$)")
axes[0].legend()

axes[1].errorbar(
    delta_summary["phase_margin"],
    delta_summary["mean_delta"],
    yerr=delta_summary["sd_delta"],
    marker="o",
    lw=1.8,
    capsize=4,
)
axes[1].axhline(0.0, linestyle="--", linewidth=1.2)
axes[1].set_xlabel("Phase margin")
axes[1].set_ylabel(r"Boundary shift relative to baseline $\Delta \eta_c$")
axes[1].set_title("Global boundary sensitivity to phase-margin choice")

plt.tight_layout()

phase_margin_fig = FIG_DIR / "phase_margin_sensitivity.png"
plt.savefig(phase_margin_fig, dpi=240)
plt.close()

print("[MARGIN] profile rows =", len(phase_margin_profiles_df))
print("[MARGIN] boundary rows =", len(phase_margin_boundary_df))
print("[TABLE] saved:", phase_margin_profiles_csv)
print("[TABLE] saved:", phase_margin_boundary_csv)
print("[FIGURE] saved:", phase_margin_fig)
print(phase_margin_boundary_df.head(12).to_string(index=False))
print(delta_summary.to_string(index=False))

[MARGIN] profile rows = 10880
[MARGIN] boundary rows = 128
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\phase_margin_profiles.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\phase_margin_boundary_summary.csv
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\figures\phase_margin_sensitivity.png
 phase_margin  attack  drift  eta_c_psecure50  max_abs_dp_deta  eta_at_max_abs_dp_deta
         0.01    0.05  0.005         0.087596            106.4                  0.0875
         0.01    0.10  0.005         0.084464            112.0                  0.0850
         0.01    0.15  0.005         0.082267            123.2                  0.0825
         0.01    0.20  0.005         0.079109            120.0                  0.0800
         0.01    0.25  0.005         0.076600            112.0                  0.0775
         0.01    0.30  0.005         0.073900            107.2                  0.0725
         0.01    0.

# 10. Time-resolved scenario analysis under repeated stochastic trajectories

This cell examines representative operating regimes through repeated stochastic trajectories rather than single realizations. For each predefined scenario, the simulator is run multiple times with independent seeds, and the resulting time-resolved paths are summarized into central trajectories and uncertainty bands for the observed QBER, secrecy margin, and normalized security indicator. The analysis also records transition timing across repeated runs, allowing the scenarios to be described in terms of both endpoint behavior and temporal stability.

In [12]:
SCENARIO_REPEATS = 150

scenario_seed_rng = np.random.default_rng(SEED + 7000)
trajectory_frames = []
trial_rows = []

for sc in SCENARIOS:
    seeds = scenario_seed_rng.integers(0, 2**32 - 1, size=SCENARIO_REPEATS)

    for repeat_id, rep_seed in enumerate(seeds):
        df = simulate_trajectory(
            eta=float(sc["eta"]),
            attack=float(sc["attack"]),
            drift=float(sc["drift"]),
            shots=DEFAULT_SHOTS,
            time_horizon=TIME_HORIZON,
            seed=int(rep_seed),
        ).copy()

        df["scenario"] = sc["name"]
        df["repeat"] = int(repeat_id)
        trajectory_frames.append(df)

        trial_rows.append(
            {
                "scenario": sc["name"],
                "repeat": int(repeat_id),
                "eta": float(sc["eta"]),
                "attack": float(sc["attack"]),
                "drift": float(sc["drift"]),
                "first_transition_time": first_transition_time(df["psi"].values),
                "final_qber_obs": float(df["qber_obs"].iloc[-1]),
                "final_secrecy_margin": float(df["secrecy_margin"].iloc[-1]),
                "final_psi": float(df["psi"].iloc[-1]),
                "final_phase": str(df["phase"].iloc[-1]),
            }
        )

scenario_traj_df = pd.concat(trajectory_frames, ignore_index=True)
scenario_trial_df = pd.DataFrame(trial_rows)

scenario_time_summary_df = (
    scenario_traj_df
    .groupby(["scenario", "t"], as_index=False)
    .agg(
        qber_mean=("qber_obs", "mean"),
        qber_q05=("qber_obs", lambda x: float(np.quantile(x, 0.05))),
        qber_q50=("qber_obs", lambda x: float(np.quantile(x, 0.50))),
        qber_q95=("qber_obs", lambda x: float(np.quantile(x, 0.95))),
        margin_mean=("secrecy_margin", "mean"),
        margin_q05=("secrecy_margin", lambda x: float(np.quantile(x, 0.05))),
        margin_q50=("secrecy_margin", lambda x: float(np.quantile(x, 0.50))),
        margin_q95=("secrecy_margin", lambda x: float(np.quantile(x, 0.95))),
        psi_mean=("psi", "mean"),
        psi_q05=("psi", lambda x: float(np.quantile(x, 0.05))),
        psi_q50=("psi", lambda x: float(np.quantile(x, 0.50))),
        psi_q95=("psi", lambda x: float(np.quantile(x, 0.95))),
        p_secure=("phase", lambda s: float(np.mean(np.asarray(s) == "secure"))),
        p_transition=("phase", lambda s: float(np.mean(np.asarray(s) == "transition"))),
        p_insecure=("phase", lambda s: float(np.mean(np.asarray(s) == "insecure"))),
    )
)

def median_transition_time(values):
    values = np.asarray(values, dtype=int)
    valid = values[values >= 0]
    if len(valid) == 0:
        return -1.0
    return float(np.median(valid))

scenario_summary_df = (
    scenario_trial_df
    .groupby("scenario", as_index=False)
    .agg(
        eta=("eta", "first"),
        attack=("attack", "first"),
        drift=("drift", "first"),
        final_qber_mean=("final_qber_obs", "mean"),
        final_qber_sd=("final_qber_obs", "std"),
        final_margin_mean=("final_secrecy_margin", "mean"),
        final_margin_sd=("final_secrecy_margin", "std"),
        final_psi_mean=("final_psi", "mean"),
        final_psi_sd=("final_psi", "std"),
        p_final_secure=("final_phase", lambda s: float(np.mean(np.asarray(s) == "secure"))),
        p_final_transition=("final_phase", lambda s: float(np.mean(np.asarray(s) == "transition"))),
        p_final_insecure=("final_phase", lambda s: float(np.mean(np.asarray(s) == "insecure"))),
        p_ever_transition=("first_transition_time", lambda x: float(np.mean(np.asarray(x) >= 0))),
        median_first_transition_time=("first_transition_time", median_transition_time),
    )
)

scenario_order = [sc["name"] for sc in SCENARIOS]
scenario_summary_df["scenario"] = pd.Categorical(
    scenario_summary_df["scenario"],
    categories=scenario_order,
    ordered=True,
)
scenario_summary_df = scenario_summary_df.sort_values("scenario").reset_index(drop=True)

scenario_time_summary_df["scenario"] = pd.Categorical(
    scenario_time_summary_df["scenario"],
    categories=scenario_order,
    ordered=True,
)
scenario_time_summary_df = scenario_time_summary_df.sort_values(["scenario", "t"]).reset_index(drop=True)

scenario_traj_csv = TAB_DIR / "scenario_trajectory_replicates.csv"
scenario_time_summary_csv = TAB_DIR / "scenario_time_summary.csv"
scenario_summary_csv = TAB_DIR / "scenario_summary.csv"

scenario_traj_df.to_csv(scenario_traj_csv, index=False)
scenario_time_summary_df.to_csv(scenario_time_summary_csv, index=False)
scenario_summary_df.to_csv(scenario_summary_csv, index=False)

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.2), sharex=True)
for ax, scenario_name in zip(axes.flat, scenario_order):
    sub = scenario_time_summary_df[scenario_time_summary_df["scenario"] == scenario_name]
    ax.plot(sub["t"], sub["psi_mean"], lw=2.0)
    ax.fill_between(sub["t"], sub["psi_q05"], sub["psi_q95"], alpha=0.18)
    ax.axhline(0.0, linestyle="--", linewidth=1.2)
    ax.set_title(scenario_name.replace("_", " "))
    ax.set_xlabel("Time")
    ax.set_ylabel(r"Normalized margin $\psi$")
plt.tight_layout()

scenario_psi_fig = FIG_DIR / "scenario_psi_profiles.png"
plt.savefig(scenario_psi_fig, dpi=240)
plt.close()

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.2), sharex=True)
for ax, scenario_name in zip(axes.flat, scenario_order):
    sub = scenario_time_summary_df[scenario_time_summary_df["scenario"] == scenario_name]
    ax.plot(sub["t"], sub["qber_mean"], lw=2.0)
    ax.fill_between(sub["t"], sub["qber_q05"], sub["qber_q95"], alpha=0.18)
    ax.axhline(QBER_SECURITY_THRESHOLD, linestyle="--", linewidth=1.2)
    ax.set_title(scenario_name.replace("_", " "))
    ax.set_xlabel("Time")
    ax.set_ylabel("Observed QBER")
plt.tight_layout()

scenario_qber_fig = FIG_DIR / "scenario_qber_profiles.png"
plt.savefig(scenario_qber_fig, dpi=240)
plt.close()

print("[SCENARIO] replicate rows =", len(scenario_traj_df))
print("[SCENARIO] trial rows =", len(scenario_trial_df))
print("[TABLE] saved:", scenario_traj_csv)
print("[TABLE] saved:", scenario_time_summary_csv)
print("[TABLE] saved:", scenario_summary_csv)
print("[FIGURE] saved:", scenario_psi_fig)
print("[FIGURE] saved:", scenario_qber_fig)
print(scenario_summary_df.to_string(index=False))

[SCENARIO] replicate rows = 72000
[SCENARIO] trial rows = 600
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\scenario_trajectory_replicates.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\scenario_time_summary.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\scenario_summary.csv
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\figures\scenario_psi_profiles.png
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\figures\scenario_qber_profiles.png
         scenario   eta  attack  drift  final_qber_mean  final_qber_sd  final_margin_mean  final_margin_sd  final_psi_mean  final_psi_sd  p_final_secure  p_final_transition  p_final_insecure  p_ever_transition  median_first_transition_time
  low_disturbance 0.035    0.08  0.008         0.056422       0.002339           0.336888         0.018990        3.062618      0.172641        1.000000            0.000000               0.

# 11. Tables and run summary

This cell assembles compact summary tables from the main results already computed in the notebook. It extracts the principal boundary estimates, robustness summaries, and scenario-level outcomes into clean, exportable tables and records a concise run summary describing the analysis scope and the key output files generated during execution. The purpose of this cell is organizational rather than inferential: it consolidates the study outputs into a form suitable for presentation, supplementary material, and archiving in a repository.

In [13]:
boundary_table_df = (
    boundary_df.loc[
        :,
        [
            "attack",
            "drift",
            "eta_c_margin",
            "eta_c_psecure50",
            "eta_c_boot_low",
            "eta_c_boot_med",
            "eta_c_boot_high",
            "max_abs_dp_deta",
        ],
    ]
    .copy()
    .sort_values(["drift", "attack"])
    .reset_index(drop=True)
)

shot_robustness_df = (
    shot_boundary_df.groupby("shots", as_index=False)
    .agg(
        mean_eta_c=("eta_c_psecure50", "mean"),
        sd_eta_c=("eta_c_psecure50", "std"),
        min_eta_c=("eta_c_psecure50", "min"),
        max_eta_c=("eta_c_psecure50", "max"),
    )
    .sort_values("shots")
    .reset_index(drop=True)
)

phase_margin_robustness_df = delta_summary.copy().sort_values("phase_margin").reset_index(drop=True)

scenario_table_df = (
    scenario_summary_df.loc[
        :,
        [
            "scenario",
            "eta",
            "attack",
            "drift",
            "final_qber_mean",
            "final_margin_mean",
            "final_psi_mean",
            "p_final_secure",
            "p_final_transition",
            "p_final_insecure",
            "p_ever_transition",
            "median_first_transition_time",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

boundary_table_csv = TAB_DIR / "table_boundary_estimates.csv"
shot_robustness_csv = TAB_DIR / "table_shot_count_robustness.csv"
phase_margin_robustness_csv = TAB_DIR / "table_phase_margin_robustness.csv"
scenario_table_csv = TAB_DIR / "table_scenario_summary.csv"

boundary_table_df.to_csv(boundary_table_csv, index=False)
shot_robustness_df.to_csv(shot_robustness_csv, index=False)
phase_margin_robustness_df.to_csv(phase_margin_robustness_csv, index=False)
scenario_table_df.to_csv(scenario_table_csv, index=False)

generated_files = sorted(str(path) for path in PROJECT_ROOT.rglob("*") if path.is_file())

summary_lines = [
    "BB84 finite-key phase study: run summary",
    "=" * 60,
    f"Run ID: {RUN_ID}",
    f"Project root: {PROJECT_ROOT}",
    "",
    "Study scope:",
    "- Operational boundary estimation from repeated endpoint simulations",
    "- Bootstrap uncertainty for secure-state probability boundary",
    "- Robustness checks across shot count and descriptive phase margin",
    "- Repeated stochastic scenario analysis with temporal uncertainty bands",
    "",
    "Key tables:",
    f"- {boundary_table_csv}",
    f"- {shot_robustness_csv}",
    f"- {phase_margin_robustness_csv}",
    f"- {scenario_table_csv}",
    "",
    "Key figures:",
    f"- {FIG_DIR / 'phase_boundary_with_uncertainty.png'}",
    f"- {FIG_DIR / 'transition_profiles_and_susceptibility.png'}",
    f"- {FIG_DIR / 'shot_count_boundary_dependence.png'}",
    f"- {FIG_DIR / 'phase_margin_sensitivity.png'}",
    f"- {FIG_DIR / 'scenario_psi_profiles.png'}",
    f"- {FIG_DIR / 'scenario_qber_profiles.png'}",
    "",
    "Interpretation:",
    "This run characterizes an operational secure-to-insecure boundary in a BB84-inspired finite-key model under disturbance, adversarial pressure, and implementation drift. Boundary estimates are obtained from repeated finite-sample simulations, uncertainty is quantified by bootstrap resampling, and robustness is examined with respect to shot count and descriptive phase-margin choice.",
    "",
    "Generated files:",
]
summary_lines.extend(f"- {fp}" for fp in generated_files)

summary_path = OTH_DIR / "study_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

print("[TABLE] saved:", boundary_table_csv)
print("[TABLE] saved:", shot_robustness_csv)
print("[TABLE] saved:", phase_margin_robustness_csv)
print("[TABLE] saved:", scenario_table_csv)
print("[SUMMARY] saved:", summary_path)

print("\n[BOUNDARY TABLE]")
print(boundary_table_df.head(10).to_string(index=False))

print("\n[SHOT ROBUSTNESS]")
print(shot_robustness_df.to_string(index=False))

print("\n[PHASE-MARGIN ROBUSTNESS]")
print(phase_margin_robustness_df.to_string(index=False))

print("\n[SCENARIO TABLE]")
print(scenario_table_df.to_string(index=False))

[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\table_boundary_estimates.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\table_shot_count_robustness.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\table_phase_margin_robustness.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\tables\table_scenario_summary.csv
[SUMMARY] saved: Outputs\bb84_finite_key_phase_study_20260423_011933\others\study_summary.txt

[BOUNDARY TABLE]
 attack  drift  eta_c_margin  eta_c_psecure50  eta_c_boot_low  eta_c_boot_med  eta_c_boot_high  max_abs_dp_deta
   0.05  0.005      0.087717         0.087237        0.086738        0.087252         0.087781            104.0
   0.10  0.005      0.084732         0.084412        0.084133        0.084427         0.084922            113.6
   0.15  0.005      0.082298         0.082045        0.081699        0.082055         0.082466            126.4
   0.20  0.005      0.